### Install Sempy

Relevant refferences: 
- [Sempy release blog](https://blog.fabric.microsoft.com/en-us/blog/semantic-link-use-fabric-notebooks-and-power-bi-datasets-for-machine-learning-data-validation-and-more/)
- [Sempy Documentation and tutorials](https://learn.microsoft.com/en-us/fabric/data-science/semantic-link-overview)

In [1]:
# Install sempy package
%pip install -U semantic-link
%pip install semantic-link-labs
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs import directlake

# Load extension 
%load_ext sempy

StatementMeta(, 22939087-4769-4e7b-b764-7f9ad076aad5, 17, Finished, Available, Finished)


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Define variables
_SemanticModelName = "LiveDemo"
_workspace = "Dive into DirectLake"

StatementMeta(, 22939087-4769-4e7b-b764-7f9ad076aad5, 19, Finished, Available, Finished)

# Get dictionary temperature - execute DMV
This in specific queries a DMV operation to get the current data in cache. 

In [8]:
%%dax "LiveDemo"

select DIMENSION_NAME, COLUMN_ID, DICTIONARY_SIZE, DICTIONARY_TEMPERATURE, DICTIONARY_LAST_ACCESSED
from $SYSTEM.DISCOVER_STORAGE_TABLE_COLUMNS
order by DICTIONARY_TEMPERATURE desc

StatementMeta(, 22939087-4769-4e7b-b764-7f9ad076aad5, 24, Finished, Available, Finished)

,DIMENSION_NAME,COLUMN_ID,DICTIONARY_SIZE,DICTIONARY_TEMPERATURE,DICTIONARY_LAST_ACCESSED
0,InternetSales,SalesAmount (51),0,18.629475,2025-11-29 10:52:08.166667
1,Product,Color (26),0,10.769748,2025-11-29 10:52:08.166667
2,InternetSales,SalesOrderNumber (36),0,8.777972,2025-11-29 10:50:45.020000
3,Product,ProductKey (20),0,1.822855,2025-11-29 10:52:08.120000
4,InternetSales,ProductKey (39),0,1.822855,2025-11-29 10:52:08.120000
5,Product,StandardCost (25),0,<NA>,NaT
6,Product,EnglishProductName (24),0,<NA>,NaT
7,Product,ProductAlternateKey (21),0,<NA>,NaT
8,Product,SizeRange (29),0,<NA>,NaT
9,Product,Size (28),0,<NA>,NaT


# Refresh operations

In [5]:
# Full model refresh - evict all data from cache! Be mindful that directlake will not work after running this command! 

fabric.refresh_dataset(
    workspace=_workspace,
    dataset=_SemanticModelName, 
    refresh_type= 'clearValues'
)
# docs: https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric?view=semantic-link-python#sempy-fabric-refresh-dataset

StatementMeta(, f76293b2-2f95-4a89-9488-b056cef02a8f, 14, Finished, Available, Finished)

'730778a6-faf0-4d16-9bcf-19c0baa536be'

In [7]:
# List the refresh requests to check if the refresh finished.
fabric.list_refresh_requests(dataset=_SemanticModelName, workspace=_workspace)

StatementMeta(, f76293b2-2f95-4a89-9488-b056cef02a8f, 16, Finished, Available, Finished)

,Id,Request Id,Start Time,End Time,Refresh Type,Service Exception Json,Status,Extended Status,Refresh Attempts
0,1612928878,730778a6-faf0-4d16-9bcf-19c0baa536be,2025-11-27 16:46:55.693000+00:00,2025-11-27 16:47:00.260000+00:00,ViaEnhancedApi,NaN,Completed,Completed,"[{'attemptId': 1, 'startTime': '2025-11-27T16:..."
1,1612928842,65a80c32-2459-09da-b55b-6f947f106db9,2025-11-27 16:46:21.063000+00:00,2025-11-27 16:46:24.853000+00:00,WebModeling,NaN,Completed,Completed,"[{'attemptId': 1, 'startTime': '2025-11-27T16:..."
2,1612867760,f3442819-8b4d-423f-ab15-6108b42bb970,2025-11-27 15:42:09.763000+00:00,2025-11-27 15:42:09.810000+00:00,DirectLakeFraming,NaN,Completed,NaN,[]
3,1612867437,8458b0d7-8a23-9a06-4446-f8cd162dbc09,2025-11-27 15:42:04.410000+00:00,2025-11-27 15:42:09.200000+00:00,WebModeling,NaN,Completed,Completed,"[{'attemptId': 1, 'startTime': '2025-11-27T15:..."


In [14]:
# back to ready mode - after this refresh directlake will work again! 
fabric.refresh_dataset(
    workspace=_workspace,
    dataset=_SemanticModelName, 
    refresh_type= 'automatic'
)

StatementMeta(, 63073b6a-43c6-4779-b698-00db1c88a3b9, 18, Finished, Available, Finished)

FabricHTTPException: 400 Bad Request for url: https://wabi-west-europe-e-primary-redirect.analysis.windows.net/v1.0/myorg/groups/62093ad9-d736-4a4a-a328-3801bd26f2fa/datasets/901e52d3-db93-4c45-a620-db0dcd6a709f/refreshes
Error: {"error":{"code":"InvalidRequest","message":"Invalid dataset refresh request. Another refresh request is already executing"}}
Headers: {'Cache-Control': 'no-store, must-revalidate, no-cache', 'Pragma': 'no-cache', 'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json; charset=utf-8', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'RequestId': 'fa11bdab-b523-48be-a216-7a489f32d7cf', 'Access-Control-Expose-Headers': 'RequestId', 'Date': 'Thu, 06 Feb 2025 13:42:11 GMT'}

In [29]:
# Refresh execution details - check for individual refresh status. 
fabric.get_refresh_execution_details(_SemanticModelName, '851a435b-ddd0-498b-b7ca-fb42dea0bad8', _workspace)

StatementMeta(, 71879c14-d5a5-4736-ba9e-1fb0609fe518, 79, Finished, Available, Finished)

RefreshExecutionDetails(start_time=datetime.datetime(2025, 2, 6, 10, 15, 33, 353000, tzinfo=datetime.timezone.utc), end_time=datetime.datetime(2025, 2, 6, 10, 15, 59, 563000, tzinfo=datetime.timezone.utc), type='Automatic', commit_mode='Transactional', status='Completed', extended_status='Completed', current_refresh_type='Automatic', number_of_attempts=0, objects=           Table      Partition     Status
0        Product        Product  Completed
1  InternetSales  InternetSales  Completed
2           Date           Date  Completed, messages=Empty DataFrame
Columns: [Message, Type]
Index: [], refresh_attempts=   Attempt Id                       Start Time  \
0           1 2025-02-06 10:15:33.861336+00:00   

                          End Time  Type  
0 2025-02-06 10:15:59.564481+00:00     0  )

# Refresh specific tables

In [10]:
# Objects to refresh, define using a dictionary
objects_to_refresh = [
    {
        "table": "InternetSales"
    },
    {
        "table": "Product"
    }
]

# Refresh the dataset
fabric.refresh_dataset(
    workspace=_workspace,
    dataset=_SemanticModelName, 
    objects=objects_to_refresh,
    refresh_type= 'automatic'
)

StatementMeta(, f76293b2-2f95-4a89-9488-b056cef02a8f, 19, Finished, Available, Finished)

'76dd198f-c8e3-4495-822d-ca668d79ca19'

### Execute Query
Below element executes queries to load the columns related in memory

In [57]:
df = fabric.evaluate_measure(
    # dataset name
    _SemanticModelName,
    # measures
    ["[$ Sales]"], 
    # groupby
    ["Product[Class]"]
)
df

StatementMeta(, 551649c1-0069-48ab-9eb5-c22a592b30b4, 16, Finished, Available, Finished)

,Class,$ Sales
0,<NA>,861774.86
1,H,22340591.8294
2,L,2133761.2238
3,M,4022558.4675


# Sempy labs

In [ ]:
# Check potential unsupported items in semantic model
labs.directlake.show_unsupported_direct_lake_objects(dataset= _SemanticModelName, workspace= _workspace)

StatementMeta(, 71879c14-d5a5-4736-ba9e-1fb0609fe518, 77, Finished, Available, Finished)

(Empty DataFrame
 Columns: [Table Name, Type]
 Index: [],
 Empty DataFrame
 Columns: [Table Name, Column Name, Type, Data Type, Source]
 Index: [],
 Empty DataFrame
 Columns: [From Table, From Column, To Table, To Column, From Column Data Type, To Column Data Type]
 Index: [])

In [11]:
# Check fallback reason
labs.directlake.check_fallback_reason(dataset= _SemanticModelName, workspace= _workspace)

StatementMeta(, f76293b2-2f95-4a89-9488-b056cef02a8f, 20, Finished, Available, Finished)

,Table Name,FallbackReasonID,Fallback Reason Detail
0,Product,0,No reason for fallback
1,InternetSales,0,No reason for fallback
2,<NA>,1,This table is not framed
